# Phase 2: Feature Engineering & SQL Database Loading
## Project: End-to-End Business Intelligence System
**Role:** Data Engineering Lead
**Objective:** Read the cleaned data, engineer new metrics/features for analysis, and load the final dataset into a PostgreSQL `bi_project` database.

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import psycopg2

# Set display options
pd.set_option('display.max_columns', None)

### Step 1: Load Staged Data
We load the clean CSV that we generated in Pipeline 1. By doing this, we keep our pipelines modular.

In [ ]:
df = pd.read_csv('cleaned_superstore.csv')

# When loading from CSV, datetimes revert to strings. We just force them back to datetime once.
if 'order_date' in df.columns:
    df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
if 'ship_date' in df.columns:
    df['ship_date'] = pd.to_datetime(df['ship_date'], errors='coerce')

print(f"Loaded Staged Data. Shape: {df.shape}")

### Step 2: Feature Engineering
Creating valuable new columns like `order_year`, `order_month`, `processing_time`, and financial parameters for our Business Analyst (Person 2).

In [ ]:
# 2.1 Date Extractions
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_quarter'] = df['order_date'].dt.quarter

# Processing Time: How long did it take to ship orders?
df['processing_days'] = (df['ship_date'] - df['order_date']).dt.days

# 2.2 Financial Engineering
# The Superstore dataset typically has 'profit' and 'sales' columns.
if 'profit' in df.columns and 'sales' in df.columns:
    df['profit_margin'] = (df['profit'] / df['sales']).fillna(0).round(4)
    # Creating a quick categorical profit indicator
    df['is_profitable'] = df['profit'] > 0

print("Feature Engineering Complete. Snapshot of new columns:")
df[['order_date', 'order_year', 'processing_days', 'profit_margin', 'is_profitable']].head()

### Step 3: SQL Database Connection & Data Loading
Using our SQLAlchemy engine to connect to our local `bi_project` PostgreSQL database and saving the final table.

In [ ]:
# Database credentials
db_user = 'postgres'    # Default user is usually postgres
db_password = 'nosqldontcry'
db_host = 'localhost'
db_port = '5432'
db_name = 'bi_project'

# Create the SQLAlchemy connection engine string
connection_url = f'postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
engine = create_engine(connection_url)

try:
    # Test the connection
    with engine.connect() as connection:
        print("Successfully connected to PostgreSQL database: bi_project!")
except Exception as e:
    print(f"Connection failed: {e}")

In [ ]:
# Save Final DataFrame to SQL Table
# This will be the table that your BI Dashboard Developer connects to via Power BI!
table_name = 'cleaned_sales_data'

try:
    # if_exists='replace' will drop the table if it already exists, 'append' adds to it.
    df.to_sql(table_name, engine, if_exists='replace', index=False)
    print(f"SUCCESS: Loaded {len(df)} rows into the '{table_name}' table in PostgreSQL!")
except Exception as e:
    print(f"Failed to load data: {e}")